# Local post-analysis: squeezing $\omega_1$ scan

Load one trajectory-resolved MCWF artifact per $\omega_1$ value from this analysis's `output/` directory, run the shared generator-moment post-analysis, and compare group-resolved and full-system squeezing.

In [ ]:
from pathlib import Path
import sys

import numpy as np


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "common").is_dir() and (candidate / "slurm").is_dir():
            return candidate
    raise RuntimeError("Could not find the repository root.")


repo_root = find_repo_root(Path.cwd().resolve())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

analysis_dir = (
    repo_root / "slurm" / "analyses" / "squeezing_omega1-scan"
)
input_dir = analysis_dir / "output"

from common.plotting import plot_squeezing
from parser.mcwf import TrajectoryEnsemble
from parser.moments import MomentSeries, SimulationMetadata
from post_analysis import (
    build_dressed_basis_transformation,
    compute_gell_mann_rotation,
    compute_j_moments,
    compute_mean_generator_moments,
    compute_s_moments,
    compute_squeezing,
    rotate_generator_moments,
)
from slurm.trajectory_ensemble_io import load_trajectory_ensemble_artifact

%reload_ext autoreload
%autoreload 2

## Input artifacts

Set `run_name` to the SLURM job name used when generating the four files. The array-task suffix is fixed by the one-task-per-$\omega_1$ simulation layout.

In [ ]:
run_name = "v4"
omega_1_values = (1.0, 0.9, 0.8, 0.7)

artifact_paths = {
    omega_1: input_dir / f"{run_name}_omega_{omega_1:.1f}_{task_index}.pkl"
    for task_index, omega_1 in enumerate(omega_1_values)
}
missing_paths = tuple(
    path for path in artifact_paths.values() if not path.is_file()
)
if missing_paths:
    raise FileNotFoundError(f"Missing simulation artifacts: {missing_paths}")

artifact_paths

## Generic moment and squeezing analysis

For each $\omega_1$, average complete trajectory rows first. Construct J and S from the averaged lab-frame moments, rotate those moments into the dressed frame, and then form group-resolved and full-system squeezing.

In [ ]:
moments_by_omega_1 = {}

for task_index, omega_1 in enumerate(omega_1_values):
    artifact_path = artifact_paths[omega_1]
    artifact = load_trajectory_ensemble_artifact(artifact_path)

    required_fields = {
        "ensemble", "simulation_metadata", "array_index", "ntraj"
    }
    missing_fields = required_fields.difference(artifact)
    if missing_fields:
        raise ValueError(
            f"{artifact_path} is missing fields: {sorted(missing_fields)}"
        )

    ensemble = artifact["ensemble"]
    metadata = artifact["simulation_metadata"]
    if not isinstance(ensemble, TrajectoryEnsemble):
        raise TypeError(f"{artifact_path} does not contain a TrajectoryEnsemble.")
    if not isinstance(metadata, SimulationMetadata):
        raise TypeError(f"{artifact_path} does not contain SimulationMetadata.")
    if artifact["array_index"] != task_index:
        raise ValueError(f"{artifact_path} has the wrong array-task index.")
    if artifact["ntraj"] != ensemble.moments.num_trajectories:
        raise ValueError(f"{artifact_path} has inconsistent trajectory counts.")
    if len(metadata.omega_i) != 1 or not np.isclose(
        metadata.omega_i[0], omega_1
    ):
        raise ValueError(f"{artifact_path} has the wrong omega_1 metadata.")

    moments = MomentSeries(
        t=ensemble.moments.t.copy(),
        metadata=metadata,
        generator_moments=ensemble.moments,
    )
    moments.mean_generator_moments = compute_mean_generator_moments(
        moments.generator_moments
    )
    moments.J = compute_j_moments(
        moments.mean_generator_moments,
        metadata=moments.metadata,
    )
    moments.S = compute_s_moments(
        moments.mean_generator_moments,
        moments.J,
    )

    transformation = build_dressed_basis_transformation(moments.J, moments.S)
    rotation = compute_gell_mann_rotation(transformation)
    mean_dressed_generator_moments = rotate_generator_moments(
        moments.mean_generator_moments,
        rotation,
    )
    moments.squeezing = compute_squeezing(
        mean_dressed_generator_moments,
        moments.J,
        metadata=moments.metadata,
        include_full_system=True,
    )
    moments_by_omega_1[omega_1] = moments

    print(
        f"omega_1={omega_1:g}: "
        f"{ensemble.moments.num_trajectories} trajectories, "
        f"couplings={metadata.omega_groups}"
    )

reference = moments_by_omega_1[omega_1_values[0]]
for omega_1 in omega_1_values[1:]:
    moments = moments_by_omega_1[omega_1]
    if not np.allclose(moments.t, reference.t, atol=1e-12, rtol=0.0):
        raise ValueError("All scan points must share the same saved-time grid.")
    if moments.metadata.phase_protocol != reference.metadata.phase_protocol:
        raise ValueError("All scan points must share the same phase protocol.")

## Group-resolved squeezing

Use one color family per $\omega_1$ value and one shade per physical group.

In [ ]:
group_squeezing_axes = None

for colour_family_index, omega_1 in enumerate(omega_1_values):
    moments = moments_by_omega_1[omega_1]
    group_squeezing = moments.squeezing.group_resolved
    labels = tuple(
        rf"$\omega_1={omega_1:g}$, group {group_index + 1}"
        for group_index in range(len(group_squeezing))
    )

    group_squeezing_fig, group_squeezing_axes = plot_squeezing(
        moments.t,
        tuple(series.xi2 for series in group_squeezing),
        tuple(series.lambda_min for series in group_squeezing),
        tuple(series.N_c for series in group_squeezing),
        tuple(series.excited_fraction_active for series in group_squeezing),
        labels=labels,
        colour_family_index=colour_family_index,
        axes=group_squeezing_axes,
        phase_protocol=reference.metadata.phase_protocol,
        title="Group-resolved generalized squeezing",
    )

## Full-system squeezing

Compare the squeezing of the unweighted sum of the two groups, including every cross-group covariance block.

In [ ]:
full_system_squeezing = tuple(
    moments_by_omega_1[omega_1].squeezing.full_system
    for omega_1 in omega_1_values
)
if any(series is None for series in full_system_squeezing):
    raise RuntimeError("Full-system squeezing was not computed.")

full_system_fig, full_system_axes = plot_squeezing(
    reference.t,
    tuple(series.xi2 for series in full_system_squeezing),
    tuple(series.lambda_min for series in full_system_squeezing),
    tuple(series.N_c for series in full_system_squeezing),
    tuple(
        series.excited_fraction_active for series in full_system_squeezing
    ),
    labels=tuple(rf"$\omega_1={omega_1:g}$" for omega_1 in omega_1_values),
    shade_index=3,
    phase_protocol=reference.metadata.phase_protocol,
    title="Full-system generalized squeezing",
)

## Analysis summary

In [ ]:
{
    omega_1: {
        "ntraj": (
            moments_by_omega_1[omega_1]
            .generator_moments.num_trajectories
        ),
        "completed couplings": (
            moments_by_omega_1[omega_1].metadata.omega_groups
        ),
        "minimum group xi2 [dB]": tuple(
            float(np.nanmin(series.xi2_db))
            for series in (
                moments_by_omega_1[omega_1].squeezing.group_resolved
            )
        ),
        "minimum full-system xi2 [dB]": float(
            np.nanmin(
                moments_by_omega_1[omega_1]
                .squeezing.full_system.xi2_db
            )
        ),
    }
    for omega_1 in omega_1_values
}